In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
import pickle
import pandas as pd
import numpy as np
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from itertools import product
from typing import List, Tuple, Optional
from IPython.display import clear_output
%matplotlib inline

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# desktop version
data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
models_folder = "G:\Meu Drive\mestrado_final_files\models\\NGramModel\\"

# google colab version
# data_folder = "drive/MyDrive/mestrado_final_files/data/"
# models_folder = "drive/MyDrive/mestrado_final_files/models/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

In [4]:
BATCH_SIZE = 1024
EPOCHS = 50
VOCAB_SIZE = 601
TIME_SIZE = 301

run_models = 1


In [5]:
%run 01_functions_training.ipynb
#%run drive/MyDrive/mestrado_final_files/notebooks/01_functions_training.ipynb

In [6]:
with open(data_folder + 'data_token_features.pkl', 'rb') as f:
    df_token_features = pickle.load(f)

In [7]:

list_seasons = ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23']
list_context = [1, 3, 5]
list_embedding_layer_dim = [32, 64]
list_num_neurons = [50]
list_num_layers = [3, 5]
list_dropout = [0, 0.3]
list_layernorm = [False, True]

In [8]:
model_mask_name = "{}_seasondata{}_context{}_embedding{}_numlayers{}_numneurons{}_{}_layernorm{}_setcovariables0_parameters.pth"

In [9]:
all_combinations = list(product(list_seasons, list_context, list_embedding_layer_dim, list_num_neurons, list_num_layers,
                                list_dropout, list_layernorm))

In [10]:
len(all_combinations)

240

In [11]:
i = 0
for comb in all_combinations:
    i += 1
    print(i)
    season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_HIDDEN_NEURONS, NUM_LAYERS, dropout_rate, layer_norm = comb

    print(comb)

    # ================== DADOS TREINO/DEV - MODELO TOTAL ==================
    df_ = df_token_features.query("season == @season & season_split == 'train'")

    trn_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    trn_y   = torch.tensor(np.asarray(df_["target_tokens"].to_list()), dtype = torch.int64, device = device)

    df_ = df_token_features.query("season == @season & season_split == 'dev'")

    dev_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    dev_y   = torch.tensor(np.asarray(df_["target_tokens"].to_list()), dtype = torch.int64, device = device)

    del df_
    # =======================================================================

    tp_dropout = "nodropout" if dropout_rate == 0 else "dropout" + str(dropout_rate).replace(".", "")

    token_model = NGramModel(
        context_size = CONTEXT_SIZE, 
        vocab_size = VOCAB_SIZE, 
        embedding_dim = EMBEDDING_LAYER_SIZE,
        num_hidden_neurons = NUM_HIDDEN_NEURONS,
        num_layers = NUM_LAYERS, 
        activation = nn.GELU, 
        dropout_rate = dropout_rate, 
        use_layer_norm = layer_norm)

    str_season = season.replace("-", "")

    model_filename = model_mask_name\
    .format(token_model.__class__.__name__, str_season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_LAYERS, NUM_HIDDEN_NEURONS, tp_dropout, str(layer_norm))
 
    results_filename = "results_"  + model_filename.replace("_parameters.pth", ".pkl")

    if results_filename in os.listdir(models_folder):
        print("Already fitted.")
        continue

    token_model, trnloss, devloss = train_ngram_model(
        model = token_model,
        train_data = trn_tkn[:, -CONTEXT_SIZE:],
        train_targets = trn_y,
        batch_size = BATCH_SIZE, 
        n_epochs = EPOCHS, 
        lr = 0.002,
        val_data = dev_tkn[:, -CONTEXT_SIZE:],
        val_targets = dev_y,
        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1},
        device = device)

    df_importance = permutation_importance_ngram(
        model = token_model, 
        target = dev_y,
        tokens = dev_tkn[:, -CONTEXT_SIZE:],
        only_tokens = True,
        num_repeats = 10)
    
    print(df_importance)

    output = token_model(trn_tkn[:, -CONTEXT_SIZE:])
    baseline_loss = F.cross_entropy(output, trn_y).item()

    print(f"train data loss: {baseline_loss}")

    torch.save(token_model.state_dict(), models_folder + model_filename)

    dict_results = {
        "class_model": token_model.__class__.__name__,
        "model_filename": model_filename,
        "token_context_size": CONTEXT_SIZE,
        "token_embedding_dim": EMBEDDING_LAYER_SIZE,
        "training_epochs": EPOCHS,
        "training_batch_loss": np.asarray(trnloss),
        "train_loss": baseline_loss,
        "dev_loss": np.asarray(devloss),
        "num_parameters": get_num_parameters(token_model),
        "variable_importance": df_importance}

    with open(models_folder + results_filename, 'wb') as f:
        pickle.dump(dict_results, f)


    clear_output(wait = True) 


1
('2018-19', 1, 32, 50, 3, 0, False)
Already fitted.
2
('2018-19', 1, 32, 50, 3, 0, True)
Already fitted.
3
('2018-19', 1, 32, 50, 3, 0.3, False)
Already fitted.
4
('2018-19', 1, 32, 50, 3, 0.3, True)
Already fitted.
5
('2018-19', 1, 32, 50, 5, 0, False)
Already fitted.
6
('2018-19', 1, 32, 50, 5, 0, True)
Already fitted.
7
('2018-19', 1, 32, 50, 5, 0.3, False)
Already fitted.
8
('2018-19', 1, 32, 50, 5, 0.3, True)
Already fitted.
9
('2018-19', 1, 64, 50, 3, 0, False)
Already fitted.
10
('2018-19', 1, 64, 50, 3, 0, True)
Already fitted.
11
('2018-19', 1, 64, 50, 3, 0.3, False)
Already fitted.
12
('2018-19', 1, 64, 50, 3, 0.3, True)
Already fitted.
13
('2018-19', 1, 64, 50, 5, 0, False)
Already fitted.
14
('2018-19', 1, 64, 50, 5, 0, True)
Already fitted.
15
('2018-19', 1, 64, 50, 5, 0.3, False)
Already fitted.
16
('2018-19', 1, 64, 50, 5, 0.3, True)
Already fitted.
17
('2018-19', 3, 32, 50, 3, 0, False)
Already fitted.
18
('2018-19', 3, 32, 50, 3, 0, True)
Already fitted.
19
('2018-19

: 